# Path patterns, by example

Every example below runs on six hand-made paths of three or four events, so you
can check each answer by eye instead of trusting the output.

The language in one table:

| Token | Matches |
|---|---|
| `cart` | that event |
| `[a\|b]` | one event, either of those |
| `[^a]` | one event, anything but `a` |
| `.` | one event, any of them |
| `.*` | a run of any length, including none |
| `[^a]*` | a run of any length containing no `a` |
| `[^a\|b]*` | a run of any length containing none of those |
| `[a\|b]*` | a run of any length containing nothing but those |
| `path_start`, `path_end` | the path's own boundaries |

Tokens are joined with `->`. Two tokens **not** separated by a gap must be
strictly adjacent.

In [1]:
import warnings

import pandas as pd

from retentioneering.eventstream import Eventstream
from retentioneering.exceptions import PatternSyntaxError, InvalidParameterError
from retentioneering.paths import anchors

PATHS = {
    "p1": ["home", "cart", "buy"],
    "p2": ["cart", "home", "cart", "buy"],
    "p3": ["home", "chat", "cart", "buy"],
    "p4": ["cart", "err", "buy"],
    "p5": ["home", "cart"],
    "p6": ["buy", "home", "cart", "buy"],
}


def build(paths):
    rows, ts = [], pd.Timestamp("2024-01-01")
    for pid, events in paths.items():
        for i, event in enumerate(events):
            rows.append(
                {
                    "user_id": pid,
                    "event": event,
                    "timestamp": ts + pd.Timedelta(minutes=i),
                }
            )
    return Eventstream(pd.DataFrame(rows))


toy = build(PATHS)
pd.DataFrame({"path": list(PATHS), "events": [" -> ".join(v) for v in PATHS.values()]})

,path,events
0,p1,home -> cart -> buy
1,p2,cart -> home -> cart -> buy
2,p3,home -> chat -> cart -> buy
3,p4,cart -> err -> buy
4,p5,home -> cart
5,p6,buy -> home -> cart -> buy


## The viewer

`show()` prints every path with the matched positions marked `<0>`, `<1>`, …
The `start` / `end` columns are the boundary sentinels — they are always there,
whether or not a pattern mentions them, which is exactly what several of the
rules below turn on.

It calls `anchors.resolve_anchors`, the engine every public pattern parameter
sits on top of. The public surface (`step_matrix`, `matches_pattern`,
`truncate_paths`) only ever asks it narrower questions; we call it directly so
the *positions* are visible, not just the yes/no.

In [2]:
def show(pattern, occurrence="first", paths=PATHS, stream=None, quiet=False):
    """Print each path with the pattern's matched positions marked."""
    stream = stream or toy
    normalized = anchors.normalize_pattern(pattern, warn=False)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore" if quiet else "always")
        match = anchors.resolve_anchors(
            stream.df, stream.schema, normalized, occurrence=occurrence
        )

    print(f"{pattern!r}   occurrence={occurrence}")
    for pid, events in paths.items():
        rows = match.frame[match.frame["user_id"] == pid]
        hits = {int(r.step): int(r.ordinal) for r in rows.itertuples()}
        labelled = [(0, "start")] + [(i + 1, e) for i, e in enumerate(events)]
        labelled += [(len(events) + 1, "end")]
        cells = [f"{name}<{hits[s]}>" if s in hits else name for s, name in labelled]
        print(f"   {pid}  {'MATCH' if hits else '     '}   " + "  ".join(cells))
    return match

---
## 1. Adjacency and gaps

`cart->buy` needs the two events **next to each other**. `cart->.*->buy` only
needs them in that order, with anything (or nothing) in between.

In [3]:
_ = show("cart->buy")

'cart->buy'   occurrence=first
   p1  MATCH   start  home  cart<0>  buy<1>  end
   p2  MATCH   start  cart  home  cart<0>  buy<1>  end
   p3  MATCH   start  home  chat  cart<0>  buy<1>  end
   p4          start  cart  err  buy  end
   p5          start  home  cart  end
   p6  MATCH   start  buy  home  cart<0>  buy<1>  end


`p4` is `cart, err, buy` — the `err` breaks the adjacency, so it does not match.
`p5` never buys at all.

In [4]:
_ = show("cart->.*->buy")

'cart->.*->buy'   occurrence=first
   p1  MATCH   start  home  cart<0>  buy<1>  end
   p2  MATCH   start  cart<0>  home  cart  buy<1>  end
   p3  MATCH   start  home  chat  cart<0>  buy<1>  end
   p4  MATCH   start  cart<0>  err  buy<1>  end
   p5          start  home  cart  end
   p6  MATCH   start  buy  home  cart<0>  buy<1>  end


Now `p4` matches: the gap swallows the `err`. Note `p1`, where the gap matched
**nothing** — `cart` and `buy` are already adjacent, and a gap of zero events is
still a gap.

`p2` and `p6` show something else: the anchor moved. With adjacency required,
`p2`'s match had to use the *second* `cart`; with a gap, the *first* one already
works. That is `occurrence="first"` at work, and section 6 comes back to it.

---
## 2. Positions

Every token except a gap takes one **position**, numbered from 0. Those numbers
are what `at=` addresses and what Step Matrix centres on.

In [5]:
for pattern in ["cart->buy", "cart->.*->buy", "home->.*->cart->buy"]:
    print(f"{pattern:26} positions -> {anchors.literal_tokens(pattern)}")

cart->buy                  positions -> ['cart', 'buy']
cart->.*->buy              positions -> ['cart', 'buy']
home->.*->cart->buy        positions -> ['home', 'cart', 'buy']


`.*` is absent from all three lists. It is not a position — it says what may sit
*between* positions.

---
## 3. One position, several events: `[a|b]`

Instead of merging events with [`rename_events`](https://retentioneering.com/docs/data-processors/rename-events) just to ask one question, list
them.

In [6]:
_ = show("home->cart")

'home->cart'   occurrence=first
   p1  MATCH   start  home<0>  cart<1>  buy  end
   p2  MATCH   start  cart  home<0>  cart<1>  buy  end
   p3          start  home  chat  cart  buy  end
   p4          start  cart  err  buy  end
   p5  MATCH   start  home<0>  cart<1>  end
   p6  MATCH   start  buy  home<0>  cart<1>  buy  end


In [7]:
_ = show("[home|chat]->cart")

'[home|chat]->cart'   occurrence=first
   p1  MATCH   start  home<0>  cart<1>  buy  end
   p2  MATCH   start  cart  home<0>  cart<1>  buy  end
   p3  MATCH   start  home  chat<0>  cart<1>  buy  end
   p4          start  cart  err  buy  end
   p5  MATCH   start  home<0>  cart<1>  end
   p6  MATCH   start  buy  home<0>  cart<1>  buy  end


The only difference is `p3` (`home, chat, cart, buy`): its `cart` follows a
`chat`, which the class now admits. Everything else is unchanged.

This is exactly what `rename_events({"home": "X", "chat": "X"})` + the pattern
`X->cart` would give you — without touching the data:

In [8]:
merged = build(
    {k: ["X" if e in ("home", "chat") else e for e in v] for k, v in PATHS.items()}
)
by_class = show("[home|chat]->cart")
by_rename = anchors.resolve_anchors(merged.df, merged.schema, "X->cart")
print("same paths:", set(by_class.paths()) == set(by_rename.paths()))

'[home|chat]->cart'   occurrence=first
   p1  MATCH   start  home<0>  cart<1>  buy  end
   p2  MATCH   start  cart  home<0>  cart<1>  buy  end
   p3  MATCH   start  home  chat<0>  cart<1>  buy  end
   p4          start  cart  err  buy  end
   p5  MATCH   start  home<0>  cart<1>  end
   p6  MATCH   start  buy  home<0>  cart<1>  buy  end
same paths: True


---
## 4. One position, anything but: `[^a]` and `.`

In [9]:
_ = show("cart->[^buy]")

'cart->[^buy]'   occurrence=first
   p1          start  home  cart  buy  end
   p2  MATCH   start  cart<0>  home<1>  cart  buy  end
   p3          start  home  chat  cart  buy  end
   p4  MATCH   start  cart<0>  err<1>  buy  end
   p5          start  home  cart  end
   p6          start  buy  home  cart  buy  end


"A cart that was **not** followed by a buy." Only `p2` (`cart -> home`) and `p4`
(`cart -> err`) qualify.

Look at what did **not** match: `p5` is `home, cart` — its `cart` is the last
event. There is no next event, and the `end` sentinel does not count as one.
That is the rule of the next section, and it is the difference between this
pattern meaning what you want and meaning nothing.

In [10]:
_ = show(".->cart")

'.->cart'   occurrence=first
   p1  MATCH   start  home<0>  cart<1>  buy  end
   p2  MATCH   start  cart  home<0>  cart<1>  buy  end
   p3  MATCH   start  home  chat<0>  cart<1>  buy  end
   p4          start  cart  err  buy  end
   p5  MATCH   start  home<0>  cart<1>  end
   p6  MATCH   start  buy  home<0>  cart<1>  buy  end


`.` is "any one event". So `.->cart` reads **"a cart that was not the first
event of the path"** — `p4` drops out because its only `cart` is the very first
event, with nothing but the `start` sentinel before it. `p2` starts with a `cart`
too, but it has a second one later, and that one qualifies.

Before classes existed this question had no pattern at all; you counted it in
pandas.

---
## 5. Boundaries are not events

`path_start` and `path_end` are real tokens you can name:

In [11]:
_ = show("path_start->cart")

'path_start->cart'   occurrence=first
   p1          start  home  cart  buy  end
   p2  MATCH   start<0>  cart<1>  home  cart  buy  end
   p3          start  home  chat  cart  buy  end
   p4  MATCH   start<0>  cart<1>  err  buy  end
   p5          start  home  cart  end
   p6          start  buy  home  cart  buy  end


In [12]:
_ = show("cart->path_end")

'cart->path_end'   occurrence=first
   p1          start  home  cart  buy  end
   p2          start  cart  home  cart  buy  end
   p3          start  home  chat  cart  buy  end
   p4          start  cart  err  buy  end
   p5  MATCH   start  home  cart<0>  end<1>
   p6          start  buy  home  cart  buy  end


But `.` and `[^...]` **never** match them, mirroring regex, where `.` does not
match a string boundary because a boundary is not a character. A sentinel takes
part only when you name it.

Compare the two — same shape, opposite answers:

In [13]:
_ = show("path_start->cart")  # cart AT the start
_ = show(".->cart")  # cart NOT at the start

'path_start->cart'   occurrence=first
   p1          start  home  cart  buy  end
   p2  MATCH   start<0>  cart<1>  home  cart  buy  end
   p3          start  home  chat  cart  buy  end
   p4  MATCH   start<0>  cart<1>  err  buy  end
   p5          start  home  cart  end
   p6          start  buy  home  cart  buy  end


'.->cart'   occurrence=first
   p1  MATCH   start  home<0>  cart<1>  buy  end
   p2  MATCH   start  cart  home<0>  cart<1>  buy  end
   p3  MATCH   start  home  chat<0>  cart<1>  buy  end
   p4          start  cart  err  buy  end
   p5  MATCH   start  home<0>  cart<1>  end
   p6  MATCH   start  buy  home<0>  cart<1>  buy  end


And the rule falls out of the definition rather than being special-cased, so
`[^path_start]` means exactly the same thing as `.`:

In [14]:
a = show("[^path_start]->cart", quiet=True)
b = show(".->cart")
print("identical:", a.frame.equals(b.frame))

'[^path_start]->cart'   occurrence=first
   p1  MATCH   start  home<0>  cart<1>  buy  end
   p2  MATCH   start  cart  home<0>  cart<1>  buy  end
   p3  MATCH   start  home  chat<0>  cart<1>  buy  end
   p4          start  cart  err  buy  end
   p5  MATCH   start  home<0>  cart<1>  end
   p6  MATCH   start  buy  home<0>  cart<1>  buy  end
'.->cart'   occurrence=first
   p1  MATCH   start  home<0>  cart<1>  buy  end
   p2  MATCH   start  cart  home<0>  cart<1>  buy  end
   p3  MATCH   start  home  chat<0>  cart<1>  buy  end
   p4          start  cart  err  buy  end
   p5  MATCH   start  home<0>  cart<1>  end
   p6  MATCH   start  buy  home<0>  cart<1>  buy  end
identical: True


---
## 6. Which match, when there are several

A pattern usually fits a path in more than one way. `occurrence` picks:

- `"first"` puts every token as **early** as it can be *in any valid match*
- `"last"` as **late** as it can be

In [15]:
_ = show("cart->.*->buy", "first")
_ = show("cart->.*->buy", "last")

'cart->.*->buy'   occurrence=first
   p1  MATCH   start  home  cart<0>  buy<1>  end
   p2  MATCH   start  cart<0>  home  cart  buy<1>  end
   p3  MATCH   start  home  chat  cart<0>  buy<1>  end
   p4  MATCH   start  cart<0>  err  buy<1>  end
   p5          start  home  cart  end
   p6  MATCH   start  buy  home  cart<0>  buy<1>  end


'cart->.*->buy'   occurrence=last
   p1  MATCH   start  home  cart<0>  buy<1>  end
   p2  MATCH   start  cart  home  cart<0>  buy<1>  end
   p3  MATCH   start  home  chat  cart<0>  buy<1>  end
   p4  MATCH   start  cart<0>  err  buy<1>  end
   p5          start  home  cart  end
   p6  MATCH   start  buy  home  cart<0>  buy<1>  end


Watch `p2` (`cart, home, cart, buy`): `first` anchors on the first `cart`,
`last` on the second. Both are valid matches of the same pattern.

The words "in any valid match" are load-bearing. `last` is **not** "the last
occurrence of the event":

In [16]:
trap = build({"t1": ["home", "cart", "buy", "cart"]})
_ = show(
    "cart->.*->buy", "last", paths={"t1": ["home", "cart", "buy", "cart"]}, stream=trap
)

'cart->.*->buy'   occurrence=last
   t1  MATCH   start  home  cart<0>  buy<1>  cart  end


The final `cart` has no `buy` after it, so it takes part in no complete match and
is not a candidate at all. `last` lands on the *first* `cart`.

---
## 7. Restricting what lies between: `[^a]*`

Quantify a class with `*` and it stops being a position — it becomes a **gap**
that says what may appear in the run between the two anchors around it.

In [17]:
_ = show("cart->.*->buy")  # any route
_ = show("cart->[^err]*->buy")  # a route that never hit an error

'cart->.*->buy'   occurrence=first
   p1  MATCH   start  home  cart<0>  buy<1>  end
   p2  MATCH   start  cart<0>  home  cart  buy<1>  end
   p3  MATCH   start  home  chat  cart<0>  buy<1>  end
   p4  MATCH   start  cart<0>  err  buy<1>  end
   p5          start  home  cart  end
   p6  MATCH   start  buy  home  cart<0>  buy<1>  end


'cart->[^err]*->buy'   occurrence=first
   p1  MATCH   start  home  cart<0>  buy<1>  end
   p2  MATCH   start  cart<0>  home  cart  buy<1>  end
   p3  MATCH   start  home  chat  cart<0>  buy<1>  end
   p4          start  cart  err  buy  end
   p5          start  home  cart  end
   p6  MATCH   start  buy  home  cart<0>  buy<1>  end


`p4` (`cart, err, buy`) is the one that drops out: it is the only path whose run
from `cart` to `buy` contains an `err`.

Two things to notice:

- `p1` still matches. The gap matched **nothing**, and an empty run contains no
  `err`, so it is clean by definition.
- `p6` is `buy, home, cart, buy` — the `err` question never arises, but note the
  match uses the second `buy`, since the first one is before the `cart`.

A gap's class takes several members just as a positional one does, so
`[^chat|err]*` is "reached `buy` from `cart` without hitting **either** support
or an error".

And the class can be positive instead — "nothing **but** these". Both need their
own paths to show up clearly, because on `PATHS` almost every route from `cart`
to `buy` is already adjacent, and an empty run satisfies any restriction:

In [18]:
CHECKOUT = {
    "q1": ["cart", "ship", "pay", "buy"],  # checkout steps only
    "q2": ["cart", "ship", "chat", "buy"],  # detoured to support
    "q3": ["cart", "buy"],  # nothing in between at all
    "q4": ["cart", "ship", "err", "buy"],  # hit an error
    "q5": ["cart", "promo", "buy"],  # a detour that is neither
}
checkout = build(CHECKOUT)

_ = show("cart->[^chat|err]*->buy", paths=CHECKOUT, stream=checkout)  # blacklist
_ = show("cart->[ship|pay]*->buy", paths=CHECKOUT, stream=checkout)  # whitelist

'cart->[^chat|err]*->buy'   occurrence=first
   q1  MATCH   start  cart<0>  ship  pay  buy<1>  end
   q2          start  cart  ship  chat  buy  end
   q3  MATCH   start  cart<0>  buy<1>  end
   q4          start  cart  ship  err  buy  end
   q5  MATCH   start  cart<0>  promo  buy<1>  end


'cart->[ship|pay]*->buy'   occurrence=first
   q1  MATCH   start  cart<0>  ship  pay  buy<1>  end
   q2          start  cart  ship  chat  buy  end
   q3  MATCH   start  cart<0>  buy<1>  end
   q4          start  cart  ship  err  buy  end
   q5          start  cart  promo  buy  end


Both forms drop `q2` (a `chat` in the way) and `q4` (an `err`), and both keep
`q1` and `q3` — `q3` on the empty run.

`q5` is the one that separates them. Its detour is a `promo`:

- `[^chat|err]*` **admits** it — a `promo` is neither of the two things ruled out;
- `[ship|pay]*` **rejects** it — a `promo` is not one of the two things allowed.

That is the whole difference in practice. A negated gap is a blacklist and stays
true as your event vocabulary grows; a positive gap is a whitelist and gets
stricter with every new event you start tracking. Pick by which list you can
actually keep complete — usually "the few things that must not happen" rather
than "everything that may".

### A gap is not a position

It takes no ordinal, so `at=`, `occurrence=` and Step Matrix centring count
exactly what they counted before:

In [19]:
for pattern in ["cart->.*->buy", "cart->[^err]*->buy", "cart->[^err]->buy"]:
    print(f"{pattern:26} positions -> {anchors.literal_tokens(pattern)}")

cart->.*->buy              positions -> ['cart', 'buy']
cart->[^err]*->buy         positions -> ['cart', 'buy']
cart->[^err]->buy          positions -> ['cart', '[^err]', 'buy']


The third line is the one to look at. `[^err]` without the `*` is a **position** —
"exactly one event, and it isn't an err" — while `[^err]*` is a run of any
length. One character, two different questions:

In [20]:
_ = show("cart->[^err]->buy")  # EXACTLY one event in between
_ = show("cart->[^err]*->buy")  # ANY number of events in between

'cart->[^err]->buy'   occurrence=first
   p1          start  home  cart  buy  end
   p2          start  cart  home  cart  buy  end
   p3          start  home  chat  cart  buy  end
   p4          start  cart  err  buy  end
   p5          start  home  cart  end
   p6          start  buy  home  cart  buy  end
'cart->[^err]*->buy'   occurrence=first
   p1  MATCH   start  home  cart<0>  buy<1>  end
   p2  MATCH   start  cart<0>  home  cart  buy<1>  end
   p3  MATCH   start  home  chat  cart<0>  buy<1>  end
   p4          start  cart  err  buy  end
   p5          start  home  cart  end
   p6  MATCH   start  buy  home  cart<0>  buy<1>  end


The first one matches nothing at all, and that is the demonstration. No path here
has exactly one event between a `cart` and a `buy` *except* `p4`, whose single
in-between event is the `err` the class rules out. Paths like `p1`, where `cart`
and `buy` are adjacent, fail it too — zero events is not one event.

The second matches four paths, because a gap is happy with zero.

### The subtle one: a restricted gap changes which occurrence qualifies

With a plain `.*`, an earlier anchor is always the safer choice — the gap does
not care how long it gets. A restricted gap inverts that: the earlier the
anchor, the longer the run that has to stay clean.

In [21]:
TRAP = {"t1": ["A", "X", "A", "D"]}
trap = build(TRAP)

_ = show("A->.*->D", paths=TRAP, stream=trap)
_ = show("A->[^X]*->D", paths=TRAP, stream=trap)

'A->.*->D'   occurrence=first
   t1  MATCH   start  A<0>  X  A  D<1>  end


'A->[^X]*->D'   occurrence=first
   t1  MATCH   start  A  X  A<0>  D<1>  end


Same path, same `occurrence="first"`, different `A`.

Under `.*` the first `A` reaches `D` fine, so "as early as it can be" is
position 1. Under `[^X]*` the first `A`'s run to `D` crosses the `X`, so that
`A` takes part in **no** valid match — and "as early as it can be *in any valid
match*" lands on the second one. The definition never changed; the set of valid
matches did.

### A restricted gap needs an anchor on both sides

At either end of a pattern its outer side is unpinned, and since a gap also
matches the empty run, it would silently mean nothing. That is refused:

In [22]:
for pattern in ["[^buy]*->cart", "cart->[^buy]*"]:
    try:
        anchors.normalize_pattern(pattern, warn=False)
    except PatternSyntaxError as exc:
        print(f"{pattern!r}\n   {exc.message}\n")

'[^buy]*->cart'
   '[^buy]*' at the start of '[^buy]*->cart' has nothing on its outer side to bound it, so it would match the empty run and mean nothing. Anchor it — e.g. 'path_start->[^buy]*->cart'.

'cart->[^buy]*'
   '[^buy]*' at the end of 'cart->[^buy]*' has nothing on its outer side to bound it, so it would match the empty run and mean nothing. Anchor it — e.g. 'cart->[^buy]*->path_end'.



Naming a boundary is usually what was meant — and gives a genuinely useful
pattern:

In [23]:
_ = show("path_start->[^buy]*->path_end")

'path_start->[^buy]*->path_end'   occurrence=first
   p1          start  home  cart  buy  end
   p2          start  cart  home  cart  buy  end
   p3          start  home  chat  cart  buy  end
   p4          start  cart  err  buy  end
   p5  MATCH   start<0>  home  cart  end<1>
   p6          start  buy  home  cart  buy  end


"Never bought": only `p5` (`home, cart`).

---
## 8. Patterns in the public API

Everything above is the engine. Here are the three places you actually type a
pattern.

### `matches_pattern` — a 0/1 metric per path

In [24]:
toy.get_metrics(
    [
        {"metric": "matches_pattern", "metric_args": {"pattern": "cart->[^err]*->buy"}},
        {"metric": "matches_pattern", "metric_args": {"pattern": ".->cart"}},
        {"metric": "matches_pattern", "metric_args": {"pattern": "[home|chat]->cart"}},
    ]
)

,matches_pattern_cart->[^err]*->buy,matches_pattern_.->cart,matches_pattern_[home|chat]->cart
user_id,,,
p1,True,True,True
p2,True,True,True
p3,True,True,True
p4,False,False,False
p5,False,True,True
p6,True,True,True


### `filter_paths` — keep whole paths

In [25]:
kept = toy.filter_paths(
    {
        "op": "=",
        "metric": "matches_pattern",
        "value": True,
        "metric_args": {"pattern": "cart->[^err]*->buy"},
    }
)
print("kept:", sorted(kept.df["user_id"].unique()))

kept: ['p1', 'p2', 'p3', 'p6']


### `truncate_paths` — cut each path to a window

`at=` addresses the pattern's positions, so the window can open on the event
that *completes* a sequence rather than on any occurrence of it.

In [26]:
window = toy.truncate_paths(start_anchor="cart", end_anchor="buy")
for pid, group in window.df.groupby("user_id", observed=True):
    print(f"   {pid}  {' -> '.join(group['event'])}")

   p1  cart -> buy
   p2  cart -> home -> cart -> buy
   p3  cart -> buy
   p4  cart -> err -> buy
   p6  cart -> buy


In [27]:
# at=0 anchors on the FIRST position of the pattern instead of the last
window = toy.truncate_paths(
    start_anchor={"pattern": "[home|chat]->cart", "at": 0}, end_anchor="path_end"
)
for pid, group in window.df.groupby("user_id", observed=True):
    print(f"   {pid}  {' -> '.join(group['event'])}")

   p1  home -> cart -> buy
   p2  home -> cart -> buy
   p3  chat -> cart -> buy
   p5  home -> cart
   p6  home -> cart -> buy


`p4` (`cart, err, buy`) is gone: it has no `cart` preceded by a `home` or `chat`,
so the window never opens.

### `step_matrix` — centre the columns on a pattern

A class fills one position, so centring works exactly as it does on a single
event. The one visible difference: column `0` holds a distribution over the
class instead of one event at `1.0`.

In [28]:
matrix = toy.step_matrix_data(path_pattern="[buy|chat]", max_steps=2)[0]
matrix.round(2)

step,-2,-1,0,1,2
event,,,,,
path_start,0.4,0.2,0.0,0.0,0.0
buy,0.0,0.0,0.8,0.0,0.2
cart,0.2,0.4,0.0,0.2,0.2
chat,0.0,0.0,0.2,0.0,0.0
err,0.0,0.2,0.0,0.0,0.0
home,0.4,0.2,0.0,0.2,0.0
path_end,0.0,0.0,0.0,0.6,0.6


Column `0` is a **distribution** — four paths anchor on a `buy` and one (`p3`)
on a `chat` — where a single-event pattern would put `1.0` on one row.

A class does not always split, though, and the reason is worth seeing:

In [29]:
toy.step_matrix_data(path_pattern="[home|chat]", max_steps=2)[0].round(2)

step,-2,-1,0,1,2
event,,,,,
path_start,1.0,0.6,0.0,0.0,0.0
buy,0.0,0.2,0.0,0.0,0.6
cart,0.0,0.2,0.0,0.8,0.2
chat,0.0,0.0,0.0,0.2,0.0
home,0.0,0.0,1.0,0.0,0.0
path_end,0.0,0.0,0.0,0.0,0.2


All `home`, no `chat`. The anchor is one *position*, and `occurrence="first"`
puts it as early as it can go — `p3` is `home, chat, cart, buy`, so its earliest
home-or-chat is the `home`. The class chose which events are eligible; it did not
make the anchor land on more than one of them per path.

A restricted gap separates blocks just as `.*` does — one block per part:

In [30]:
blocks = toy.step_matrix_data(path_pattern="cart->[^err]*->buy", max_steps=2)
print(f"{len(blocks)} blocks")
for i, block in enumerate(blocks):
    print(f"\n--- block {i} ---")
    print(block.round(2))

2 blocks

--- block 0 ---
step          -2    -1   0     1     2 
event                                  
path_start  0.50  0.25  0.0  0.00  0.00
buy         0.25  0.00  0.0  0.75  0.00
cart        0.00  0.00  1.0  0.00  0.25
chat        0.00  0.25  0.0  0.00  0.00
home        0.25  0.50  0.0  0.25  0.00
path_end    0.00  0.00  0.0  0.00  0.75

--- block 1 ---
step          -2   -1   0    1    2 
event                               
path_start  0.00  0.0  0.0  0.0  0.0
buy         0.00  0.0  1.0  0.0  0.0
cart        0.00  1.0  0.0  0.0  0.0
chat        0.25  0.0  0.0  0.0  0.0
home        0.75  0.0  0.0  0.0  0.0
path_end    0.00  0.0  0.0  1.0  1.0


---
## 9. What the parser refuses, and why

Negation and alternation are supported **exactly where their scope is bounded**:
one position, or a run pinned by an anchor on each side. Everything wider needs
an automaton, and the engine underneath is relational.

In [31]:
events_of_toy = toy.df["event"].unique().tolist()

BAD = [
    "[^cart->buy]",  # negating a sequence
    "cart->buy|home->cart",  # branching
    "(cart|buy)",  # round brackets
    "^cart",  # regex start anchor
    "cart$",  # regex end anchor
    "[^[home|chat]]",  # nested brackets
    "[]",  # empty class
    "[home]+",  # a quantifier other than *
    "cart->.*->[^err]*->buy",  # two gaps in a row
]
for pattern in BAD:
    try:
        # Shape is settled by the parser; "did you mean" hints need the event
        # vocabulary, so they surface in validation. Both are one user-visible
        # step, so check both here.
        normalized = anchors.normalize_pattern(pattern, warn=False)
        anchors.validate_pattern_tokens(normalized, events_of_toy, param="path_pattern")
        print(f"{pattern!r}\n   (accepted)\n")
    except PatternSyntaxError as exc:
        print(f"{pattern!r}\n   {exc.message}\n")

'[^cart->buy]'
   A class in [ ] describes one event, not a sequence — '[^cart->buy]' contains '->'.

'cart->buy|home->cart'
   '|' only works inside brackets, where it means one event that is any of several: '[buy|home]'. A pattern cannot list alternative sequences.

'(cart|buy)'
   Round brackets are not supported. For one event that is any of several, write '[cart|buy]'.

'^cart'
   '^' is not a start-of-path anchor. To match the beginning of a path write 'path_start->...'; to match one event that is not 'cart' write '[^cart]'.

'cart$'
   '$' is not an end-of-path anchor. Write '...->path_end' instead.

'[^[home|chat]]'
   Nested brackets in '[^[home|chat]]'. A class is a flat list of event names — write '[^home|chat]'.

'[]'
   Empty class '[]' — list at least one event name.

'[home]+'
   '[home]+': only '*' (any number of events, including none) is supported after a class.

'cart->.*->[^err]*->buy'
   Two gaps in a row in 'cart->.*->[^err]*->buy' ('.*' then '[^err]*'). A gap alr

### Typos are caught, including inside a class

This matters more for a class than for a plain name. A typo in `by` matches
nothing and gives you an empty result — annoying but obvious. A typo in `[^by]`
excludes nothing and gives you an **always-true** position: a wrong answer that
looks like a healthy one.

In [32]:
events = toy.df["event"].unique().tolist()
for pattern in ["cart->by", "cart->[^by]", "cart->[^by]*->home"]:
    try:
        anchors.validate_pattern_tokens(
            anchors.normalize_pattern(pattern, warn=False), events, param="path_pattern"
        )
    except InvalidParameterError as exc:
        print(f"{pattern!r}\n   {exc.message}\n")

'cart->by'
   Invalid value 'by' for parameter 'path_pattern'. Allowed values: ['buy', 'cart', 'chat', 'err', 'home']

'cart->[^by]'
   Invalid value 'by' for parameter 'path_pattern'. Allowed values: ['buy', 'cart', 'chat', 'err', 'home']

'cart->[^by]*->home'
   Invalid value 'by' for parameter 'path_pattern'. Allowed values: ['buy', 'cart', 'chat', 'err', 'home']



### A pattern with nothing to look for

Legal, but almost always true — so it warns rather than silently returning
everything. A *positive* class counts as something to look for; only negation
and wildcards widen.

In [33]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    anchors.validate_pattern_tokens("[^cart]->.*->.", events, param="path_pattern")
    anchors.validate_pattern_tokens(
        "[home|chat]->.*->buy", events, param="path_pattern"
    )
for w in caught:
    print(w.message)
print(f"\n{len(caught)} warning(s) — the second pattern is anchored enough")

path_pattern '[^cart]->.*->.' names no events to look for — every position is negated or a wildcard, so it matches almost any path. Add at least one event name or a positive class to anchor it.

1 warning(s) — the second pattern is anchored enough


---
## Cheat sheet

| You want | Write |
|---|---|
| B right after A | `A->B` |
| B eventually after A | `A->.*->B` |
| exactly one event between A and B | `A->.->B` |
| B after A, never touching X | `A->[^X]*->B` |
| B after A, touching neither X nor Y | `A->[^X\|Y]*->B` |
| B after A, through nothing but X and Y | `A->[X\|Y]*->B` |
| A or B in one position | `[A\|B]` |
| anything but A in one position | `[^A]` |
| A was the path's first event | `path_start->A` |
| A was **not** the path's first event | `.->A` |
| the path never contained A | `path_start->[^A]*->path_end` |
| the path ended right after A | `A->path_end` |

Full reference: the **Path Patterns** page in the docs.